# Letterform Classification Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ipavlopoulos/diachronic-greek-letterforms/blob/main/Inference.ipynb)

This notebook loads the released CNN checkpoint and predicts the Greek letter class for one sample cliplet.

GitHub shows a static preview. To run the notebook, open it in Colab or local Jupyter.

## Setup

When running in Colab, this cell clones the repository so that `source.py`, `best_cnn_letter_model.pth`, and the sample cliplets are available.

In [ ]:
import os
import random
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path("source.py").exists():
    repo_dir = Path("/content/diachronic-greek-letterforms")
    if not repo_dir.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/ipavlopoulos/diachronic-greek-letterforms.git",
            str(repo_dir),
        ], check=True)
    os.chdir(repo_dir)

sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())

## Select an Image

The model expects a cropped image containing one Greek letterform. This example samples from `data/palitchar/cliplets`.

In [ ]:
from PIL import Image
from IPython.display import display

cliplet_dir = Path("data/palitchar/cliplets")
image_paths = sorted(
    path for path in cliplet_dir.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

selected_path = random.choice(image_paths)
print(selected_path)
display(Image.open(selected_path).convert("L").resize((160, 160)))

## Predict the Letter

The image is converted to grayscale, resized to `64x64`, normalized, and passed through the released `CNN2D` checkpoint.

In [ ]:
import torch

from source import LETTER_LABELS, image_path_to_tensor, load_letterform_model

model = load_letterform_model("best_cnn_letter_model.pth", device="cpu")
image_tensor = image_path_to_tensor(selected_path, device="cpu")

with torch.no_grad():
    logits = model(image_tensor)
    probabilities = torch.softmax(logits, dim=1)[0]
    predicted_index = int(probabilities.argmax())

print("Predicted class:", LETTER_LABELS[predicted_index])
print("Confidence:", round(float(probabilities[predicted_index]), 3))